# Nighttime Lens Flare Removal — Experiment 3 (Attention U-Net)

This is a separate, from-scratch Attention U-Net experiment. It preserves the verified Experiment 2 dataset download, extraction, official Flare7K loader, preprocessing, disjoint background split, and fixed validation construction. It changes the model to an Attention U-Net and uses a balanced reconstruction objective without the low-flare preservation penalty.

**Experiment 2 remains untouched.** All Experiment 3 outputs are written to `/kaggle/working/flare_unet_experiment_3/`.

**Backup preflight:** Before starting a new Kaggle session, verify that `Flare_Removal_Experiment2_Complete.zip` and `idse8_experiment2.ipynb` have both been downloaded and copied to Google Drive. Kaggle may only allow one active session, and a new notebook does not inherit the old session's `/kaggle/working/` files.

Run this notebook from the top in a fresh Kaggle GPU session. The dataset archives are downloaded from the same Google Drive file IDs used by the earlier experiments. No Baseline 1 or Experiment 2 weights are imported: this model is intentionally trained from scratch.

**Extended run for today:** up to 20 epochs, early-stopping patience 6. The original 4,000/500 train/validation background counts are intentionally unchanged. A portable `Experiment3_RESUME_LATEST.zip` is refreshed during training; its FileLink is displayed once and points to the latest refreshed archive. Download it periodically and keep it in Google Drive in case the Kaggle session is lost.

## 1. Environment and configuration

Enable **GPU T4** in Kaggle Settings before running. In the right-side **Settings → Session options → Persistence**, select **Files only** before starting this run if that option is available. This can help preserve files under `/kaggle/working` across normal interactive-session restarts, but it is best-effort and does not preserve GPU memory or guarantee recovery after a hard crash.

This notebook stores checkpoints, logs and datasets under `/kaggle/working`. A full local training state is saved every 100 batches. A portable recovery ZIP is refreshed every 500 batches and at each epoch boundary. The first refresh displays a FileLink; that same link points to the latest refreshed ZIP as training continues. Download it periodically and upload/replace the backup in Google Drive. For a fresh Kaggle session, paste the Drive share link or file ID into `RESUME_BUNDLE_DRIVE_LINK` in Cell 3 before running the notebook. The restore cell downloads and extracts the checkpoint automatically.

In [ ]:
import os, sys, time, random, zipfile, shutil, subprocess, json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms

In [ ]:
# Reproducibility and Experiment 3 configuration
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# This experiment is intended for Kaggle GPU (T4), not CPU training.
assert torch.cuda.is_available(), (
    "CUDA GPU is not available. In Kaggle, open Settings → Accelerator → GPU, "
    "then restart the session before continuing."
)
device = torch.device("cuda")
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

WORK = Path("/kaggle/working")
DATA_DIR = WORK / "flare_data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Same Google Drive archive IDs as Baseline 1 and Experiment 2.
FLICKR_FILE_ID = "1mRPLMijWWt9Q-_6-f7wYgSyskPM6vb4G"
FLARE_FILE_ID = "1tJbK97sNLakOll6CJYU66AO_9E0xvA_0"
FLICKR_ZIP = DATA_DIR / "Flickr24K.zip"
FLARE_ZIP = DATA_DIR / "Flare7K++.zip"

# Keep the same input size, batch size, and background split limits.
IMAGE_SIZE = 256
BATCH_SIZE = 4
TRAIN_LIMIT = 4000
VAL_LIMIT = 500
EPOCHS = 20
LEARNING_RATE = 1e-4
NUM_WORKERS = 0

# Reconstruction objective: ordinary L1 protects the full image; spatially
# weighted L1 gives additional emphasis to pixels with larger corruption.
FLARE_WEIGHT = 10.0
ORDINARY_L1_WEIGHT = 0.7
WEIGHTED_L1_WEIGHT = 0.3

# No low-flare preservation term in this experiment. Experiment 2's term
# penalised predicted corrections in the lowest 40% of residual pixels, which
# may conflict with the objective of correcting weak flare. This is an explicit
# objective change and is recorded in the experiment metadata.

CHECKPOINT_EVERY_BATCHES = 100
# A portable recovery ZIP is refreshed every 500 training batches and at each epoch end.
RESUME_BUNDLE_EVERY_BATCHES = 500
EARLY_STOPPING_PATIENCE = 6

# Optional: paste a Google Drive share URL or file ID for a previously downloaded
# Experiment 3 resume bundle when continuing in a NEW Kaggle session. Leave blank
# for a fresh run or when the latest checkpoint is already in /kaggle/working.
RESUME_BUNDLE_DRIVE_LINK = ""

EXPERIMENT_NAME = "experiment_3_attention_unet"
ARCHITECTURE_NAME = "Attention U-Net (compact 16-256 channels)"
RUN_DIR = WORK / "flare_unet_experiment_3"
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Experiment:", EXPERIMENT_NAME)
print("Dataset directory:", DATA_DIR)
print("Run directory:", RUN_DIR)
print("Training backgrounds:", TRAIN_LIMIT)
print("Validation backgrounds:", VAL_LIMIT)
print("Epoch ceiling:", EPOCHS, "| early-stopping patience:", EARLY_STOPPING_PATIENCE)
print("Learning rate:", LEARNING_RATE)
print("Resume bundle Drive link provided:", bool(RESUME_BUNDLE_DRIVE_LINK.strip()))

## 2. Install download and evaluation utilities

Kaggle's PyTorch, torchvision, NumPy and matplotlib are already supplied by the image. We install only `gdown` for Google Drive downloads and `scikit-image` for SSIM/PSNR evaluation.

In [ ]:
%pip install -q gdown scikit-image
import gdown
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
print("gdown and scikit-image are ready.")

## 3. Download the datasets

This cell checks existing files first. If a ZIP already exists and is a valid archive, it will not download it again. If Google Drive permissions prevent download, make sure both files are accessible to anyone with the link.

In [ ]:
def valid_zip(path):
    return path.exists() and path.stat().st_size > 1_000_000 and zipfile.is_zipfile(path)

def download_drive_file(file_id, output_path):
    if valid_zip(output_path):
        print(f"Using existing archive: {output_path.name} ({output_path.stat().st_size/1024**3:.2f} GB)")
        return
    if output_path.exists():
        output_path.unlink()
    url = f"https://drive.google.com/uc?id={file_id}"
    result = gdown.download(url, str(output_path), quiet=False, fuzzy=True)
    if result is None or not valid_zip(output_path):
        raise RuntimeError(
            f"Download failed or file is not a valid ZIP: {output_path}. "
            "Check Google Drive sharing permissions and retry."
        )
    print(f"Downloaded {output_path.name}: {output_path.stat().st_size/1024**3:.2f} GB")

download_drive_file(FLICKR_FILE_ID, FLICKR_ZIP)
download_drive_file(FLARE_FILE_ID, FLARE_ZIP)

# Optional recovery path for a NEW Kaggle session. The bundle is generated by
# this notebook during training and contains latest_checkpoint.pth + best_model.pth.
if RESUME_BUNDLE_DRIVE_LINK.strip():
    resume_url = RESUME_BUNDLE_DRIVE_LINK.strip()
    if not resume_url.startswith("http"):
        resume_url = f"https://drive.google.com/uc?id={resume_url}"
    resume_archive = WORK / "Experiment3_RESUME_FROM_DRIVE.zip"
    downloaded = gdown.download(resume_url, str(resume_archive), quiet=False, fuzzy=True)
    if downloaded is None or not zipfile.is_zipfile(resume_archive):
        raise RuntimeError(
            "Could not download a valid Experiment 3 resume ZIP. Check that its "
            "Google Drive sharing is set to Anyone with the link (Viewer)."
        )
    with zipfile.ZipFile(resume_archive, "r") as zf:
        members = {Path(info.filename).name: info for info in zf.infolist() if not info.is_dir()}
        if "latest_checkpoint.pth" not in members:
            raise RuntimeError("Resume ZIP does not contain latest_checkpoint.pth")
        for filename in ["latest_checkpoint.pth", "best_model.pth"]:
            if filename in members:
                (RUN_DIR / filename).write_bytes(zf.read(members[filename]))
    print("Restored portable checkpoint files into:", RUN_DIR)
else:
    print("No Drive resume bundle supplied; local checkpoint will be used if present, otherwise training starts fresh.")

## 4. Extract and verify the archives

Extraction is skipped when the expected top-level folder is already present. This avoids re-extracting several gigabytes if the cell is rerun.

In [ ]:
def find_named_dir(root, target_name):
    root = Path(root)
    matches = [p for p in root.rglob(target_name) if p.is_dir()]
    if not matches:
        return None
    # Prefer the shallowest match.
    return sorted(matches, key=lambda p: len(p.parts))[0]

def safe_extract(zip_path, destination):
    with zipfile.ZipFile(zip_path) as zf:
        bad_member = zf.testzip()
        if bad_member is not None:
            raise RuntimeError(f"Corrupt archive member: {bad_member}")
        zf.extractall(destination)

flickr_dir = find_named_dir(DATA_DIR, "Flickr24K")
if flickr_dir is None:
    print("Extracting Flickr24K...")
    safe_extract(FLICKR_ZIP, DATA_DIR)
else:
    print("Flickr24K already extracted:", flickr_dir)

flare7k_dir = find_named_dir(DATA_DIR, "Flare7K")
if flare7k_dir is None:
    print("Extracting Flare7K++...")
    safe_extract(FLARE_ZIP, DATA_DIR)
else:
    print("Flare7K++ already extracted.")

# Re-discover after extraction; Flare7K is nested inside Flare7Kpp/Flare7K.
flickr_dir = find_named_dir(DATA_DIR, "Flickr24K")
flare7k_dir = find_named_dir(DATA_DIR, "Flare7K")
assert flickr_dir is not None, "Could not find Flickr24K folder after extraction."
assert flare7k_dir is not None, "Could not find Flare7K folder after extraction."

scattering_root = flare7k_dir / "Scattering_Flare"
reflective_root = flare7k_dir / "Reflective_Flare"
assert scattering_root.is_dir(), f"Missing {scattering_root}"
assert reflective_root.is_dir(), f"Missing {reflective_root}"

print("Flickr folder:", flickr_dir)
print("Flare7K folder:", flare7k_dir)
print("Scattering categories:", sorted(p.name for p in scattering_root.iterdir() if p.is_dir()))

## 5. Clone the official Flare7K repository

The repository is cloned into the new notebook's working directory. No previous notebook files are required.

In [ ]:
REPO_DIR = WORK / "Flare7K"
if not (REPO_DIR / "data_loader.py").exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/ykdai/Flare7K.git", str(REPO_DIR)],
        check=True
    )
assert (REPO_DIR / "data_loader.py").exists()
print("Official loader:", REPO_DIR / "data_loader.py")

## 6. Build the synthetic paired dataset

The official loader returns this tuple, in this exact order:

`clean_image, flare_image, corrupted_image, flare_mask, gamma`

It creates synthetic flare-corrupted samples on the fly. The five Flare7K scattering folders are loaded individually because the loader searches only the immediate directory passed to it. The Flare-R captured-pattern folders are also added when present in the archive.

In [ ]:
from pathlib import Path

loader_path = Path("/kaggle/working/Flare7K/data_loader.py")

# Restore the original loader from GitHub.
import subprocess

subprocess.run(
    [
        "git", "checkout", "HEAD", "--", "data_loader.py"
    ],
    cwd="/kaggle/working/Flare7K",
    check=True
)

# Confirm that the original file is valid Python.
loader_code = loader_path.read_text()
compile(loader_code, str(loader_path), "exec")

print("Original data_loader.py restored successfully.")

In [ ]:
from pathlib import Path

loader_path = Path("/kaggle/working/Flare7K/data_loader.py")
code = loader_path.read_text()

old = """\t\t\treflective_img=adjust_gamma(reflective_img)
\t\t\tflare_img = torch.clamp(flare_img+reflective_img,min=0,max=1)"""

new = """\t\t\treflective_img=adjust_gamma(reflective_img)
\t\t\tif reflective_img.shape[-2:] != flare_img.shape[-2:]:
\t\t\t\treflective_img = torch.nn.functional.interpolate(
\t\t\t\t\treflective_img.unsqueeze(0),
\t\t\t\t\tsize=flare_img.shape[-2:],
\t\t\t\t\tmode="bilinear",
\t\t\t\t\talign_corners=False
\t\t\t\t).squeeze(0)
\t\t\tflare_img = torch.clamp(flare_img+reflective_img,min=0,max=1)"""

assert old in code, "Expected code block not found. No changes made."

code = code.replace(old, new, 1)

compile(code, str(loader_path), "exec")
loader_path.write_text(code)

print("Patch applied. Syntax check passed.")

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/Flare7K")

from data_loader import Flare_Image_Loader

print("Flare_Image_Loader imported successfully.")

In [ ]:
from torchvision import transforms

base_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomCrop(IMAGE_SIZE, pad_if_needed=True),
    transforms.RandomHorizontalFlip()
])

flare_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.RandomAffine(
        degrees=(0, 360),
        scale=(0.8, 1.5),
        translate=(0.20, 0.20),
        shear=(-20, 20)
    ),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))
])

print("Both transforms defined successfully.")

In [ ]:
for name in ["flickr_dir", "base_transform", "flare_transform", "IMAGE_SIZE"]:
    print(f"{name}: {'defined' if name in globals() else 'NOT defined'}")

In [ ]:
sys.path.insert(0, str(REPO_DIR))

from data_loader import Flare_Image_Loader


# Background augmentation: resize while preserving aspect ratio, then crop.
base_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomCrop(IMAGE_SIZE, pad_if_needed=True),
    transforms.RandomHorizontalFlip()
])

# Flare augmentation: randomise scale, position, orientation and reflections.
# The official loader applies its own flare synthesis/gamma/noise operations afterwards.
flare_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.RandomAffine(
        degrees=(0, 360),
        scale=(0.8, 1.5),
        translate=(0.20, 0.20),
        shear=(-20, 20)
    ),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))
])

dataset = Flare_Image_Loader(
    image_path=str(flickr_dir),
    transform_base=base_transform,
    transform_flare=flare_transform,
    mask_type="luminance"
)

scattering_categories = [
    "Light_Source", "Compound_Flare", "Glare_with_shimmer", "Core", "Streak"
]
for category in scattering_categories:
    category_path = scattering_root / category
    assert category_path.is_dir(), f"Missing scattering category: {category_path}"
    dataset.load_scattering_flare(category, str(category_path))

dataset.load_reflective_flare("Reflective", str(reflective_root))

# Flare-R contains 962 real-captured flare patterns. Add its two pattern
# folders to the same scattering pool when the extracted archive includes them.
flare_r_root = flare7k_dir.parent / "Flare-R"
flare_r_loaded = []
for category in ["Compound_Flare", "Light_Source"]:
    category_path = flare_r_root / category
    if category_path.is_dir():
        dataset.load_scattering_flare(f"FlareR_{category}", str(category_path))
        flare_r_loaded.append(category)
print("Flare-R categories loaded:", flare_r_loaded if flare_r_loaded else "not present")

print("Background images:", len(dataset.data_list))
print("Scattering flare patterns:", len(dataset.flare_list))
print("Reflective flare patterns:", len(dataset.reflective_list))
print("Dataset length:", len(dataset))
assert len(dataset.data_list) > 0
assert len(dataset.flare_list) > 0
assert len(dataset.reflective_list) > 0

In [ ]:
sample = dataset[0]

print("Number of returned values:", len(sample))

for i, item in enumerate(sample):
    if torch.is_tensor(item):
        print(f"Output {i}: shape={tuple(item.shape)}, dtype={item.dtype}")
    else:
        print(f"Output {i}: {item}")

In [ ]:
clean_image, flare_image, corrupted_image, flare_mask, gamma = dataset[0]

input_image = corrupted_image
target_image = clean_image

print("Input shape:", input_image.shape)
print("Target shape:", target_image.shape)
print("Input range:", input_image.min().item(), input_image.max().item())
print("Target range:", target_image.min().item(), target_image.max().item())

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 4

train_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

batch = next(iter(train_loader))

clean, flare, corrupted, mask, gamma = batch

print("Clean:", clean.shape)
print("Flare:", flare.shape)
print("Corrupted:", corrupted.shape)
print("Mask:", mask.shape)
print("Gamma:", gamma.shape)

In [ ]:
print("Dataset tensors and split will be checked before the model is defined.")

## 7. Data-quality audit and visual inspection

This checks the actual tensors returned by the loader, not just filenames. Do not continue if the sample is not three-channel RGB, finite, or in the expected `[0,1]` range.

In [ ]:
def count_image_modes(folder, limit=300):
    paths = []
    for ext in ("*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif"):
        paths.extend(Path(folder).glob(ext))
    counts = Counter()
    for p in paths[:limit]:
        try:
            with Image.open(p) as im:
                counts[im.mode] += 1
        except Exception as exc:
            print("Unreadable image:", p, exc)
    return counts

print("Flickr image modes (sample):", count_image_modes(flickr_dir))
print("Scattering image modes (sample):", count_image_modes(scattering_root / "Light_Source"))
print("Reflective image modes (sample):", count_image_modes(reflective_root))

clean0, flare0, corrupted0, mask0, gamma0 = dataset[0]
for name, tensor in [
    ("clean", clean0), ("flare", flare0),
    ("corrupted", corrupted0), ("mask", mask0)
]:
    print(name, "shape:", tuple(tensor.shape), "dtype:", tensor.dtype,
          "range:", (float(tensor.min()), float(tensor.max())),
          "finite:", bool(torch.isfinite(tensor).all()))

assert clean0.shape == (3, IMAGE_SIZE, IMAGE_SIZE), "Clean sample is not RGB 256x256."
assert flare0.shape == clean0.shape and corrupted0.shape == clean0.shape
assert torch.isfinite(clean0).all() and torch.isfinite(corrupted0).all()
assert float(clean0.min()) >= 0 and float(clean0.max()) <= 1
assert float(corrupted0.min()) >= 0 and float(corrupted0.max()) <= 1

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, img, title in zip(
    axes, [clean0, corrupted0, flare0],
    ["Clean target", "Corrupted input", "Synthetic flare"]
):
    ax.imshow(img.permute(1, 2, 0).cpu().numpy().clip(0, 1))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Split backgrounds into train and validation

We split the clean-background indices before selecting the smaller experiment subset. This prevents the same background index from appearing in both sets. Flare patterns are sampled randomly by the official loader when an item is generated.

In [ ]:
all_indices = np.arange(len(dataset))
rng = np.random.default_rng(SEED)
rng.shuffle(all_indices)

validation_pool_size = max(1, int(0.20 * len(all_indices)))
validation_pool = all_indices[:validation_pool_size]
training_pool = all_indices[validation_pool_size:]

train_indices = training_pool[:min(TRAIN_LIMIT, len(training_pool))].tolist()
val_indices = validation_pool[:min(VAL_LIMIT, len(validation_pool))].tolist()

assert set(train_indices).isdisjoint(val_indices)
print("Training backgrounds:", len(train_indices))
print("Validation backgrounds:", len(val_indices))
print("Overlap:", len(set(train_indices).intersection(val_indices)))

## 9. Cache a fixed validation set and create DataLoaders

Only 500 validation samples are generated and cached. This is much faster and lighter than caching the full validation pool. Validation images remain identical between epochs.

In [ ]:
class FixedValidationDataset(Dataset):
    def __init__(self, source_dataset, indices, seed=42):
        self.samples = []
        py_state = random.getstate()
        np_state = np.random.get_state()
        torch_state = torch.random.get_rng_state()

        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        try:
            for idx in indices:
                self.samples.append(source_dataset[int(idx)])
        finally:
            random.setstate(py_state)
            np.random.set_state(np_state)
            torch.random.set_rng_state(torch_state)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

train_dataset = Subset(dataset, train_indices)
fixed_val_dataset = FixedValidationDataset(dataset, val_indices, seed=SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True
)
val_loader = DataLoader(
    fixed_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Fixed validation samples:", len(fixed_val_dataset))

## 10. Mandatory batch-order test

This is the key correctness check before training. The model input is the **third** returned tensor (`corrupted`); the target is the **first** (`clean`).

In [ ]:
clean_b, flare_b, corrupted_b, mask_b, gamma_b = next(iter(train_loader))

print("Clean:", clean_b.shape, clean_b.min().item(), clean_b.max().item())
print("Flare:", flare_b.shape, flare_b.min().item(), flare_b.max().item())
print("Corrupted:", corrupted_b.shape, corrupted_b.min().item(), corrupted_b.max().item())

assert clean_b.shape == corrupted_b.shape == (BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE)
assert torch.isfinite(clean_b).all() and torch.isfinite(corrupted_b).all()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, title in zip(
    axes, [clean_b[0], corrupted_b[0], flare_b[0]],
    ["Clean target", "Corrupted model input", "Flare pattern"]
):
    ax.imshow(img.permute(1, 2, 0).numpy().clip(0, 1))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 11. Compact Attention U-Net architecture

The encoder-decoder channel widths remain close to the compact U-Net used previously (16, 32, 64, 128, 256). The architectural change is the addition of attention gates on the four skip connections. Each gate uses the decoder feature to produce a spatial attention map for its corresponding encoder feature before concatenation.

The network predicts a three-channel flare residual, and the clean estimate remains `corrupted_input - predicted_flare`. The final prediction layer is zero-initialised so the untrained model initially returns the input unchanged.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class AttentionGate(nn.Module):
    """Spatially gates an encoder skip feature using a decoder gating feature."""
    def __init__(self, gating_channels, skip_channels, inter_channels):
        super().__init__()
        self.gating_projection = nn.Sequential(
            nn.Conv2d(gating_channels, inter_channels, kernel_size=1, bias=True),
            nn.BatchNorm2d(inter_channels)
        )
        self.skip_projection = nn.Sequential(
            nn.Conv2d(skip_channels, inter_channels, kernel_size=1, bias=True),
            nn.BatchNorm2d(inter_channels)
        )
        self.attention_coefficients = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.Conv2d(inter_channels, 1, kernel_size=1, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )

    def forward(self, skip_features, gating_features):
        gating = self.gating_projection(gating_features)
        skip = self.skip_projection(skip_features)
        if gating.shape[-2:] != skip.shape[-2:]:
            gating = F.interpolate(
                gating, size=skip.shape[-2:], mode="bilinear", align_corners=False
            )
        attention_map = self.attention_coefficients(gating + skip)
        return skip_features * attention_map


class FlareAttentionUNet(nn.Module):
    """Attention U-Net that predicts flare residual; clean = input - residual."""
    def __init__(self):
        super().__init__()
        self.pool = nn.MaxPool2d(2)

        self.enc1 = DoubleConv(3, 16)
        self.enc2 = DoubleConv(16, 32)
        self.enc3 = DoubleConv(32, 64)
        self.enc4 = DoubleConv(64, 128)
        self.bottleneck = DoubleConv(128, 256)

        self.up4 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.att4 = AttentionGate(128, 128, 64)
        self.dec4 = DoubleConv(256, 128)

        self.up3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.att3 = AttentionGate(64, 64, 32)
        self.dec3 = DoubleConv(128, 64)

        self.up2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.att2 = AttentionGate(32, 32, 16)
        self.dec2 = DoubleConv(64, 32)

        self.up1 = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2)
        self.att1 = AttentionGate(16, 16, 8)
        self.dec1 = DoubleConv(32, 16)

        self.out = nn.Conv2d(16, 3, kernel_size=1)
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        s4 = self.att4(e4, d4)
        d4 = self.dec4(torch.cat([d4, s4], dim=1))

        d3 = self.up3(d4)
        s3 = self.att3(e3, d3)
        d3 = self.dec3(torch.cat([d3, s3], dim=1))

        d2 = self.up2(d3)
        s2 = self.att2(e2, d2)
        d2 = self.dec2(torch.cat([d2, s2], dim=1))

        d1 = self.up1(d2)
        s1 = self.att1(e1, d1)
        d1 = self.dec1(torch.cat([d1, s1], dim=1))

        flare_estimate = self.out(d1)
        clean_estimate = x - flare_estimate
        return clean_estimate, flare_estimate


model = FlareAttentionUNet().to(device)
parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Architecture:", ARCHITECTURE_NAME)
print("Trainable parameters:", f"{parameter_count:,}")

# Shape and finite-value smoke test before allocating a training run.
with torch.no_grad():
    test_input = torch.rand(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
    test_clean, test_flare = model(test_input)
assert test_clean.shape == test_flare.shape == (2, 3, IMAGE_SIZE, IMAGE_SIZE)
assert torch.isfinite(test_clean).all() and torch.isfinite(test_flare).all()
assert torch.allclose(test_clean, test_input, atol=1e-6), (
    "Zero-initialised output layer should make the initial clean prediction equal the input."
)
print("Attention U-Net forward-pass test passed.")

## 12. Initialise a fresh model, optimizer and checkpoint paths

Experiment 3 is trained **from scratch**. It does not download or load Baseline 1 or Experiment 2 weights. The model, Adam optimizer, cosine learning-rate scheduler and AMP scaler all start fresh.

The new output directory prevents this run from overwriting any earlier experiment. Its checkpoint configuration also records the architecture and loss weights, so incompatible settings cannot be silently resumed.

In [ ]:
criterion = nn.L1Loss()  # Ordinary L1 is retained for validation/checkpoint selection.
optimizer = Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

BEST_CHECKPOINT = RUN_DIR / "best_model.pth"
LATEST_CHECKPOINT = RUN_DIR / "latest_checkpoint.pth"
FINAL_WEIGHTS = RUN_DIR / "flare_removal_attention_unet_best_weights.pth"

print("Training from scratch: yes")
print("AMP enabled:", use_amp)
print("Learning rate:", LEARNING_RATE)
print("Epochs:", EPOCHS)
print("Loss:", f"{ORDINARY_L1_WEIGHT} * ordinary L1 + {WEIGHTED_L1_WEIGHT} * weighted L1")
print("Run directory:", RUN_DIR)
print("Latest resumable checkpoint:", LATEST_CHECKPOINT)
print("Best validation checkpoint:", BEST_CHECKPOINT)

## 13. Training and validation functions

Training objective:

`0.7 × ordinary L1 + 0.3 × spatially weighted L1`

The per-image spatial weights are computed from the actual post-loader corruption residual `(corrupted - clean)` and normalised to mean 1. Ordinary L1 retains uniform pressure across the whole image; weighted L1 increases emphasis on more strongly corrupted pixels.

The Experiment 2 low-flare preservation penalty is intentionally omitted. It penalised predicted corrections in the lowest 40% of residual-magnitude pixels, which may discourage removal of weak flare. This experiment therefore includes a loss-objective change as well as an architecture change; any improvement cannot be attributed to attention gates alone.

Validation and best-checkpoint selection continue to use ordinary L1 on the same fixed synthetic validation samples. Full training state is saved atomically every 100 batches.

In [ ]:
from torch.utils.data._utils.collate import default_collate


def capture_rng_state():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def restore_rng_state(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and state.get("cuda") is not None:
        torch.cuda.set_rng_state_all(state["cuda"])


def current_run_config():
    return {
        "experiment": EXPERIMENT_NAME,
        "architecture": ARCHITECTURE_NAME,
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "train_limit": TRAIN_LIMIT,
        "val_limit": VAL_LIMIT,
        "learning_rate": LEARNING_RATE,
        "flare_weight": FLARE_WEIGHT,
        "ordinary_l1_weight": ORDINARY_L1_WEIGHT,
        "weighted_l1_weight": WEIGHTED_L1_WEIGHT,
        "low_flare_preservation_weight": 0.0,
        "epochs": EPOCHS,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "checkpoint_every_batches": CHECKPOINT_EVERY_BATCHES,
        "resume_bundle_every_batches": RESUME_BUNDLE_EVERY_BATCHES,
        "optimizer": "Adam",
        "scheduler": "CosineAnnealingLR",
        "seed": SEED,
    }


def atomic_torch_save(payload, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, destination)


def make_checkpoint(
    current_epoch, next_batch, epoch_order, running_loss_sum, batches_done,
    history, best_val_loss, epochs_without_improvement, completed=False
):
    return {
        "format_version": 5,
        "config": current_run_config(),
        "current_epoch": int(current_epoch),
        "epoch": int(current_epoch - 1 if epoch_order is None else current_epoch),
        "val_loss": float(history["val_loss"][-1]) if history["val_loss"] else float("inf"),
        "next_batch": int(next_batch),
        "epoch_order": epoch_order,
        "running_loss_sum": float(running_loss_sum),
        "batches_done": int(batches_done),
        "history": history,
        "best_val_loss": float(best_val_loss),
        "epochs_without_improvement": int(epochs_without_improvement),
        "completed": bool(completed),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "rng_state": capture_rng_state(),
    }


def save_latest_checkpoint(**kwargs):
    atomic_torch_save(make_checkpoint(**kwargs), LATEST_CHECKPOINT)


resume_link_displayed = False

def create_resume_bundle():
    """Create a portable ZIP that can restore this run in a new Kaggle session."""
    global resume_link_displayed
    from IPython.display import FileLink, display

    if not LATEST_CHECKPOINT.exists():
        print("No latest checkpoint exists yet; resume bundle was not created.")
        return None

    bundle_path = WORK / "Experiment3_RESUME_LATEST.zip"
    temporary_path = WORK / "Experiment3_RESUME_LATEST.zip.tmp"
    with zipfile.ZipFile(temporary_path, "w", compression=zipfile.ZIP_STORED) as archive:
        archive.write(LATEST_CHECKPOINT, arcname="latest_checkpoint.pth")
        if BEST_CHECKPOINT.exists():
            archive.write(BEST_CHECKPOINT, arcname="best_model.pth")
        archive.writestr(
            "resume_config.json",
            json.dumps(current_run_config(), indent=2)
        )
    os.replace(temporary_path, bundle_path)

    print(
        "Portable resume bundle refreshed:", bundle_path,
        f"({bundle_path.stat().st_size / (1024**2):.1f} MB)"
    )
    print("Download this ZIP and upload/replace the previous resume backup in Google Drive.")
    os.chdir(str(WORK))
    if not resume_link_displayed:
        display(FileLink(
            bundle_path.name,
            result_html_prefix="<b>Download latest Experiment 3 recovery bundle: </b>"
        ))
        resume_link_displayed = True
    else:
        print("The FileLink displayed earlier in this cell output now points to this refreshed ZIP.")
    return bundle_path


def train_one_batch(batch):
    model.train()
    clean, flare, corrupted, mask, gamma = batch
    clean = clean.to(device, non_blocking=True)
    corrupted = corrupted.to(device, non_blocking=True)

    # Use the actual corruption residual after loader clipping/gamma/noise.
    actual_flare = (corrupted - clean).abs().mean(dim=1)
    weights = 1.0 + FLARE_WEIGHT * actual_flare
    weights = weights / weights.mean(dim=(1, 2), keepdim=True).clamp_min(1e-8)

    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
        clean_prediction, flare_prediction = model(corrupted)
        pixel_loss = F.l1_loss(
            clean_prediction.float(), clean, reduction="none"
        ).mean(dim=1)
        ordinary_l1 = pixel_loss.mean()
        weighted_l1 = (pixel_loss * weights).mean()
        loss = (
            ORDINARY_L1_WEIGHT * ordinary_l1
            + WEIGHTED_L1_WEIGHT * weighted_l1
        )

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    return float(loss.detach().float().item())


@torch.no_grad()
def validate_one_epoch():
    model.eval()
    running_loss = 0.0
    batch_count = 0
    for clean, flare, corrupted, mask, gamma in val_loader:
        clean = clean.to(device, non_blocking=True)
        corrupted = corrupted.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            clean_prediction, _ = model(corrupted)
            loss = criterion(clean_prediction.float(), clean)
        running_loss += loss.item()
        batch_count += 1
    return running_loss / max(batch_count, 1)


print("Checkpoint format:", 5)
print("Local checkpoint interval:", CHECKPOINT_EVERY_BATCHES, "batches")
print("Portable resume bundle interval:", RESUME_BUNDLE_EVERY_BATCHES, "batches")
print("Training objective: 0.7 ordinary L1 + 0.3 spatially weighted L1")

In [ ]:
print("Training function:", train_one_batch.__name__)
print("Architecture:", ARCHITECTURE_NAME)
print("Best checkpoint:", BEST_CHECKPOINT)
print("Latest checkpoint:", LATEST_CHECKPOINT)
print("Run directory:", RUN_DIR)

assert BEST_CHECKPOINT.parent == RUN_DIR
assert LATEST_CHECKPOINT.parent == RUN_DIR
assert CHECKPOINT_EVERY_BATCHES >= 1
assert ORDINARY_L1_WEIGHT + WEIGHTED_L1_WEIGHT == 1.0

## 14. Resumable Experiment 3 training

The Attention U-Net starts from random initialisation and trains for up to **20 epochs** with Adam at `1e-4`, cosine learning-rate decay, and mixed precision on the Kaggle GPU. Early stopping uses a patience of 6 epochs. The training and validation background counts remain 4,000 and 500 respectively, preserving the seeded split and fixed validation set used in the previous experiments.

Every 100 batches, the notebook atomically saves the full local training state. The state includes model weights, optimizer, scheduler, AMP scaler, RNG states, epoch permutation, batch position, and history. A portable recovery ZIP is refreshed every 500 batches and at each epoch boundary; download it from the FileLink shown in the training output and upload it to Google Drive. If a Kaggle session is lost, paste that Drive share link or file ID into `RESUME_BUNDLE_DRIVE_LINK` in Cell 3 of a fresh copy of this notebook. The restore code downloads the ZIP and resumes from its latest checkpoint.

The saved configuration is checked on resume. Do not change the epoch ceiling, training size, batch size, learning rate, or loss configuration after training has started; make a separate experiment for a different configuration.

In [ ]:
history = {"train_loss": [], "val_loss": []}
best_val_loss = float("inf")
epochs_without_improvement = 0
current_epoch = 1
next_batch = 0
epoch_order = None
running_loss_sum = 0.0
batches_done = 0
training_completed = False

if LATEST_CHECKPOINT.exists():
    print("Found checkpoint. Restoring training state...")
    checkpoint = torch.load(
        LATEST_CHECKPOINT, map_location="cpu", weights_only=False
    )
    if checkpoint.get("format_version") != 5:
        raise RuntimeError(
            "An incompatible checkpoint exists. Use a new RUN_DIR or remove "
            "the old checkpoint; this run will not silently load old weights."
        )
    if checkpoint["config"] != current_run_config():
        raise RuntimeError(
            "Training configuration differs from the saved checkpoint.\n"
            f"Saved: {checkpoint['config']}\n"
            f"Current: {current_run_config()}\n"
            "Keep the original settings to resume, or choose a new RUN_DIR "
            "for a separate experiment."
        )

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])

    history = checkpoint["history"]
    best_val_loss = checkpoint["best_val_loss"]
    epochs_without_improvement = checkpoint["epochs_without_improvement"]
    current_epoch = checkpoint["current_epoch"]
    next_batch = checkpoint["next_batch"]
    epoch_order = checkpoint["epoch_order"]
    running_loss_sum = checkpoint["running_loss_sum"]
    batches_done = checkpoint["batches_done"]
    training_completed = checkpoint["completed"]
    restore_rng_state(checkpoint["rng_state"])

    print(
        f"Restored epoch {current_epoch}/{EPOCHS}, "
        f"next batch {next_batch // BATCH_SIZE + 1}, "
        f"best validation L1={best_val_loss:.6f}"
    )
else:
    print("No Experiment 3 checkpoint found. Starting Attention U-Net training from scratch.")

if training_completed:
    print("Training is already complete. Skipping the training loop.")
else:
    for epoch in range(current_epoch, EPOCHS + 1):
        epoch_start = time.time()

        # Reuse the saved permutation when resuming in the middle of an epoch.
        if epoch_order is None:
            epoch_order = torch.randperm(len(train_dataset))
            next_batch = 0
            running_loss_sum = 0.0
            batches_done = 0

        epoch_order = epoch_order.cpu()
        total_samples = len(epoch_order)

        for batch_start in range(next_batch, total_samples, BATCH_SIZE):
            batch_indices = epoch_order[batch_start:batch_start + BATCH_SIZE].tolist()
            samples = [train_dataset[int(idx)] for idx in batch_indices]
            batch = default_collate(samples)

            batch_loss = train_one_batch(batch)
            running_loss_sum += batch_loss
            batches_done += 1
            next_batch = batch_start + len(batch_indices)

            if (
                batches_done % CHECKPOINT_EVERY_BATCHES == 0
                or next_batch >= total_samples
            ):
                save_latest_checkpoint(
                    current_epoch=epoch,
                    next_batch=next_batch,
                    epoch_order=epoch_order,
                    running_loss_sum=running_loss_sum,
                    batches_done=batches_done,
                    history=history,
                    best_val_loss=best_val_loss,
                    epochs_without_improvement=epochs_without_improvement,
                    completed=False,
                )
                print(
                    f"Checkpoint saved | epoch {epoch}/{EPOCHS} | "
                    f"{next_batch}/{total_samples} samples"
                )
                if (
                    batches_done % RESUME_BUNDLE_EVERY_BATCHES == 0
                    or next_batch >= total_samples
                ):
                    create_resume_bundle()

        train_loss = running_loss_sum / max(batches_done, 1)
        val_loss = validate_one_epoch()
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        improved = val_loss < best_val_loss
        if improved:
            best_val_loss = val_loss
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        stop_early = epochs_without_improvement >= EARLY_STOPPING_PATIENCE
        completed = (epoch >= EPOCHS) or stop_early

        # At an epoch boundary, next_epoch starts with a fresh permutation.
        next_epoch = epoch + 1
        epoch_order_for_save = None
        next_batch_for_save = 0

        checkpoint_args = dict(
            current_epoch=next_epoch,
            next_batch=next_batch_for_save,
            epoch_order=epoch_order_for_save,
            running_loss_sum=0.0,
            batches_done=0,
            history=history,
            best_val_loss=best_val_loss,
            epochs_without_improvement=epochs_without_improvement,
            completed=completed,
        )
        save_latest_checkpoint(**checkpoint_args)

        if improved:
            best_payload = make_checkpoint(**checkpoint_args)
            atomic_torch_save(best_payload, BEST_CHECKPOINT)

        # Refresh once more after validation/epoch-boundary state is final.
        create_resume_bundle()

        elapsed = (time.time() - epoch_start) / 60
        print(
            f"Epoch {epoch}/{EPOCHS} | training loss {train_loss:.6f} | "
            f"validation L1 {val_loss:.6f} | best validation L1 "
            f"{best_val_loss:.6f} | {elapsed:.1f} min"
        )

        # Reset in-memory epoch state before starting the next epoch.
        current_epoch = next_epoch
        next_batch = 0
        epoch_order = None
        running_loss_sum = 0.0
        batches_done = 0

        if stop_early:
            print("Early stopping: validation L1 did not improve.")
            break

    print("Training loop finished.")
    print("Latest checkpoint:", LATEST_CHECKPOINT)
    print("Best checkpoint:", BEST_CHECKPOINT)


### Recovery after manually interrupting training

If you stop the training cell manually while the Kaggle session is still alive, run the next cell before restarting or closing the session. It packages the most recent local checkpoint so you can download it and keep a copy in Google Drive.

In [ ]:
if LATEST_CHECKPOINT.exists():
    create_resume_bundle()
else:
    print("No checkpoint is available to export yet.")

In [ ]:
paths = [BEST_CHECKPOINT, LATEST_CHECKPOINT, FINAL_WEIGHTS]
for path in paths:
    print(f"{'FOUND' if path.exists() else 'MISSING'}: {path}")

In [ ]:
print("Experiment 3 output files:")
for path in sorted(RUN_DIR.iterdir()):
    if path.is_file():
        print(path.name, f"({path.stat().st_size / (1024**2):.2f} MB)")

In [ ]:
path_checks = {
    "Repository": REPO_DIR,
    "Flickr24K": flickr_dir,
    "Flare7K++": flare7k_dir,
}
for name, path in path_checks.items():
    path = Path(path)
    print(f"{name}: {'FOUND' if path.exists() else 'MISSING'} -> {path}")

## 15. Plot training and validation loss

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history["train_loss"], marker="o", label="Combined training loss")
axes[0].plot(history["val_loss"], marker="s", label="Validation L1")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)
axes[1].plot(history["val_loss"], marker="o")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation L1")
axes[1].grid(True)
plt.tight_layout()
plt.show()

## 16. Load the best checkpoint and calculate PSNR/SSIM

The best checkpoint is selected using ordinary validation L1. PSNR and SSIM are then calculated on the same fixed synthetic validation pairs, with the corrupted input reported as the reference baseline. These are image-restoration metrics, not classification accuracy.

In [ ]:
best_state = torch.load(BEST_CHECKPOINT, map_location=device, weights_only=False)
model.load_state_dict(best_state["model_state_dict"])
model.eval()
print("Best epoch:", best_state["epoch"])
print("Best validation L1:", best_state["val_loss"])


@torch.no_grad()
def evaluate_metrics(loader):
    results = {"model_psnr": [], "model_ssim": [], "input_psnr": [], "input_ssim": []}
    model.eval()
    for clean, flare, corrupted, mask, gamma in loader:
        clean_device = clean.to(device)
        corrupted_device = corrupted.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            prediction, _ = model(corrupted_device)
        prediction = prediction.float().clamp(0, 1).cpu()

        for i in range(clean.shape[0]):
            gt = clean[i].permute(1, 2, 0).numpy().clip(0, 1)
            for tag, image in (("model", prediction[i]), ("input", corrupted[i])):
                image_np = image.permute(1, 2, 0).numpy().clip(0, 1)
                results[f"{tag}_psnr"].append(
                    peak_signal_noise_ratio(gt, image_np, data_range=1.0)
                )
                results[f"{tag}_ssim"].append(
                    structural_similarity(gt, image_np, channel_axis=2, data_range=1.0)
                )
    return {key: float(np.mean(values)) for key, values in results.items()}


val_results = evaluate_metrics(val_loader)
print(f"Input baseline: PSNR {val_results['input_psnr']:.2f} dB | SSIM {val_results['input_ssim']:.4f}")
print(f"U-Net output:   PSNR {val_results['model_psnr']:.2f} dB | SSIM {val_results['model_ssim']:.4f}")

## 17. Visual comparison: corrupted input, model output, clean target

In [ ]:
@torch.no_grad()
def show_predictions(number_of_images=4):
    model.eval()
    clean, flare, corrupted, mask, gamma = next(iter(val_loader))
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
        prediction, flare_estimate = model(corrupted[:number_of_images].to(device))
    prediction = prediction.float().clamp(0, 1).cpu()

    fig, axes = plt.subplots(number_of_images, 3, figsize=(12, 4 * number_of_images))
    if number_of_images == 1:
        axes = np.expand_dims(axes, axis=0)
    for i in range(number_of_images):
        images = [corrupted[i], prediction[i], clean[i]]
        titles = ["Corrupted input", "Model output", "Clean target"]
        for j, (image, title) in enumerate(zip(images, titles)):
            axes[i, j].imshow(image.permute(1, 2, 0).numpy().clip(0, 1))
            axes[i, j].set_title(title)
            axes[i, j].axis("off")
    plt.tight_layout()
    plt.show()

show_predictions(4)

## 18. Optional real paired-test evaluation

If the extracted Flare7K++ archive contains `test_data/real/input` and `test_data/real/gt`, evaluate those pairs separately. This is a held-out real-image check; do not mix these images into training. The code matches files by stem and reports how many pairs were found.

In [ ]:
class RealPairDataset(Dataset):
    def __init__(self, input_dir, target_dir, image_size):
        allowed = {".png", ".jpg", ".jpeg", ".bmp"}
        input_files = {p.stem: p for p in Path(input_dir).iterdir()
                       if p.is_file() and p.suffix.lower() in allowed}
        target_files = {p.stem: p for p in Path(target_dir).iterdir()
                        if p.is_file() and p.suffix.lower() in allowed}
        self.keys = sorted(set(input_files) & set(target_files))
        self.input_files = input_files
        self.target_files = target_files
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, index):
        key = self.keys[index]
        with Image.open(self.target_files[key]) as im:
            clean = self.transform(im.convert("RGB"))
        with Image.open(self.input_files[key]) as im:
            corrupted = self.transform(im.convert("RGB"))
        return clean, corrupted


real_root = flare7k_dir.parent / "test_data" / "real"
real_input = real_root / "input"
real_gt = real_root / "gt"
real_results = None

if real_input.is_dir() and real_gt.is_dir():
    real_dataset = RealPairDataset(real_input, real_gt, IMAGE_SIZE)
    if len(real_dataset) > 0:
        real_loader = DataLoader(real_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
        # Evaluation helper for paired (clean, corrupted) batches.
        @torch.no_grad()
        def evaluate_real_pairs(loader):
            values = {"model_psnr": [], "model_ssim": [], "input_psnr": [], "input_ssim": []}
            model.eval()
            for clean, corrupted in loader:
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                    prediction, _ = model(corrupted.to(device))
                prediction = prediction.float().clamp(0, 1).cpu()
                for i in range(clean.shape[0]):
                    gt = clean[i].permute(1, 2, 0).numpy().clip(0, 1)
                    for label, image in (("model", prediction[i]), ("input", corrupted[i])):
                        image_np = image.permute(1, 2, 0).numpy().clip(0, 1)
                        values[f"{label}_psnr"].append(peak_signal_noise_ratio(gt, image_np, data_range=1.0))
                        values[f"{label}_ssim"].append(structural_similarity(
                            gt, image_np, channel_axis=2, data_range=1.0
                        ))
            return {k: float(np.mean(v)) for k, v in values.items()}

        real_results = evaluate_real_pairs(real_loader)
        print("Real paired test count:", len(real_dataset))
        print("Real input baseline:", real_results["input_psnr"], "dB | SSIM", real_results["input_ssim"])
        print("Real model result:  ", real_results["model_psnr"], "dB | SSIM", real_results["model_ssim"])
    else:
        print("Real test folders exist, but no matching filename stems were found.")
else:
    print("Real paired test folders not found. Skipping real-reference evaluation.")

## 19. Save the best Attention U-Net weights

In [ ]:
torch.save(model.state_dict(), FINAL_WEIGHTS)
print("Saved best Attention U-Net weights:", FINAL_WEIGHTS)
print("Best checkpoint:", BEST_CHECKPOINT)
print("Latest resumable checkpoint:", LATEST_CHECKPOINT)

In [ ]:
model.eval()
clean, flare, corrupted, mask, gamma = next(iter(val_loader))
clean = clean.to(device)
corrupted = corrupted.to(device)

with torch.no_grad(), torch.autocast(
    device_type=device.type, dtype=torch.float16, enabled=use_amp
):
    clean_pred, flare_pred = model(corrupted)

clean_pred = clean_pred.float()
flare_pred = flare_pred.float()
actual_residual = corrupted - clean

print("Actual corruption residual magnitude:", actual_residual.abs().mean().item())
print("Predicted correction magnitude:", flare_pred.abs().mean().item())
print("Input-target MAE:", (corrupted - clean).abs().mean().item())
print("Output-target MAE:", (clean_pred - clean).abs().mean().item())
print("Flare residual prediction MAE:", (flare_pred - actual_residual).abs().mean().item())

In [ ]:
print("Actual residual mean:", actual_residual.mean().item())
print("Predicted residual mean:", flare_pred.mean().item())
print("Actual residual std:", actual_residual.std().item())
print("Predicted residual std:", flare_pred.std().item())

In [ ]:
print("Training history:")
for key, values in history.items():
    print(f"\n{key}:")
    print(values)

In [ ]:
magnitudes = []
stds = []

for i in range(len(fixed_val_dataset)):
    clean_i, flare_i, corrupted_i, mask_i, gamma_i = fixed_val_dataset[i]
    actual_residual_i = corrupted_i - clean_i
    magnitudes.append(actual_residual_i.abs().mean().item())
    stds.append(actual_residual_i.std().item())

print("Fixed validation residual statistics")
print("Mean residual magnitude:", float(np.mean(magnitudes)))
print("Mean residual standard deviation:", float(np.mean(stds)))
print("Minimum residual magnitude:", float(np.min(magnitudes)))
print("Maximum residual magnitude:", float(np.max(magnitudes)))

In [ ]:
model.eval()

baseline_errors = []
model_errors = []
actual_flare_stds = []
predicted_flare_stds = []

with torch.no_grad():
    for clean, flare, corrupted, mask, gamma in val_loader:
        clean = clean.to(device)
        corrupted = corrupted.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            clean_pred, flare_pred = model(corrupted)
        clean_pred = clean_pred.float().clamp(0, 1)
        flare_pred = flare_pred.float()

        baseline_errors.append((corrupted - clean).abs().mean().item())
        model_errors.append((clean_pred - clean).abs().mean().item())
        actual_flare_stds.append((corrupted - clean).std().item())
        predicted_flare_stds.append(flare_pred.std().item())

full_validation_results = {
    "input_l1": float(np.mean(baseline_errors)),
    "model_l1": float(np.mean(model_errors)),
    "actual_residual_std": float(np.mean(actual_flare_stds)),
    "predicted_residual_std": float(np.mean(predicted_flare_stds)),
}
print("Full validation comparison:")
print(json.dumps(full_validation_results, indent=2))

In [ ]:
# Evaluate input-vs-model L1 separately for weak, medium and strong synthetic
# validation corruptions. Quantile boundaries are computed once on this fixed set.
model.eval()
records = []

with torch.no_grad():
    for clean, flare, corrupted, mask, gamma in val_loader:
        clean_device = clean.to(device)
        corrupted_device = corrupted.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            clean_pred, _ = model(corrupted_device)
        clean_pred = clean_pred.float().clamp(0, 1).cpu()

        for i in range(clean.shape[0]):
            actual_residual = corrupted[i] - clean[i]
            records.append({
                "strength": float(actual_residual.abs().mean().item()),
                "input_l1": float(actual_residual.abs().mean().item()),
                "model_l1": float((clean_pred[i] - clean[i]).abs().mean().item()),
            })

results_df = pd.DataFrame(records)
q1, q2 = np.quantile(results_df["strength"].to_numpy(), [1/3, 2/3])
results_df["flare_group"] = np.select(
    [results_df["strength"] <= q1, results_df["strength"] <= q2],
    ["Weak", "Medium"],
    default="Strong"
)

flare_group_metrics = {}
for group_name in ["Weak", "Medium", "Strong"]:
    group = results_df[results_df["flare_group"] == group_name]
    if len(group) == 0:
        flare_group_metrics[group_name] = {"count": 0}
        continue
    input_l1 = float(group["input_l1"].mean())
    model_l1 = float(group["model_l1"].mean())
    flare_group_metrics[group_name] = {
        "count": int(len(group)),
        "mean_residual_strength": float(group["strength"].mean()),
        "input_l1": input_l1,
        "model_l1": model_l1,
        "relative_l1_improvement_percent": (
            float((input_l1 - model_l1) / max(input_l1, 1e-12) * 100.0)
        ),
    }

print("Flare-strength quantile boundaries:", float(q1), float(q2))
print(pd.DataFrame(flare_group_metrics).T)
print("Positive relative improvement means model L1 is lower than input L1.")

In [ ]:
training_history_path = RUN_DIR / "training_history.json"
training_history_path.write_text(json.dumps(history, indent=2))

results = {
    "experiment": EXPERIMENT_NAME,
    "architecture": ARCHITECTURE_NAME,
    "training_strategy": "from_scratch",
    "best_epoch": int(best_state["epoch"]),
    "best_validation_L1": float(best_state["val_loss"]),
    "synthetic_validation": val_results,
    "full_validation_l1_and_residual_stats": full_validation_results,
    "flare_strength_groups": flare_group_metrics,
    "flare_strength_quantile_boundaries": {
        "weak_medium": float(q1),
        "medium_strong": float(q2),
    },
    "real_paired_test": real_results,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "train_backgrounds": len(train_indices),
    "validation_backgrounds": len(val_indices),
    "epochs_completed": len(history["train_loss"]),
    "epoch_ceiling": EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "learning_rate": LEARNING_RATE,
    "loss": {
        "ordinary_l1_weight": ORDINARY_L1_WEIGHT,
        "weighted_l1_weight": WEIGHTED_L1_WEIGHT,
        "flare_weight": FLARE_WEIGHT,
        "low_flare_preservation_weight": 0.0,
    },
    "trainable_parameters": int(parameter_count),
}

RESULTS_PATH = RUN_DIR / "results.json"
RESULTS_PATH.write_text(json.dumps(results, indent=2))
print(json.dumps(results, indent=2))
print("Saved results:", RESULTS_PATH)
print("Saved training history:", training_history_path)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

model.eval()

# Select the median-residual sample from each validation strength group.
flare_strengths = results_df["strength"].to_numpy()
selected = []
for group_name in ["Weak", "Medium", "Strong"]:
    indices = np.where(results_df["flare_group"].to_numpy() == group_name)[0]
    if len(indices) == 0:
        continue
    median_idx = indices[np.argsort(flare_strengths[indices])[len(indices) // 2]]
    selected.append((group_name, int(median_idx)))

fig, axes = plt.subplots(len(selected), 5, figsize=(20, 4 * len(selected)))
if len(selected) == 1:
    axes = np.expand_dims(axes, axis=0)

for row, (group_name, idx) in enumerate(selected):
    clean, flare, corrupted, mask, gamma = fixed_val_dataset[idx]
    with torch.no_grad(), torch.autocast(
        device_type=device.type, dtype=torch.float16, enabled=use_amp
    ):
        prediction, flare_estimate = model(corrupted.unsqueeze(0).to(device))

    prediction = prediction[0].float().clamp(0, 1).cpu()
    flare_estimate = flare_estimate[0].float().cpu()
    actual_residual = corrupted - clean
    actual_map = actual_residual.abs().mean(dim=0).numpy()
    predicted_map = flare_estimate.abs().mean(dim=0).numpy()
    vmax = max(float(actual_map.max()), float(predicted_map.max()), 1e-6)

    images = [
        corrupted.permute(1, 2, 0).numpy(),
        clean.permute(1, 2, 0).numpy(),
        prediction.permute(1, 2, 0).numpy(),
        actual_map,
        predicted_map,
    ]
    titles = [
        "Corrupted input", "Clean target", "Attention U-Net output",
        "Actual residual magnitude", "Predicted residual magnitude"
    ]

    for col, (image, title) in enumerate(zip(images, titles)):
        ax = axes[row, col]
        if col < 3:
            ax.imshow(np.clip(image, 0, 1))
        else:
            ax.imshow(image, cmap="inferno", vmin=0, vmax=vmax)
        ax.set_title(title)
        ax.axis("off")
    axes[row, 0].set_ylabel(
        f"{group_name}\nStrength: {flare_strengths[idx]:.6f}", fontsize=11
    )

plt.tight_layout()
plt.show()

In [ ]:
import zipfile
from pathlib import Path
from IPython.display import display, FileLink
import os

BACKUP_ZIP = WORK / "Experiment3_COMPLETE.zip"
files_to_backup = [
    FINAL_WEIGHTS,
    BEST_CHECKPOINT,
    LATEST_CHECKPOINT,
    RESULTS_PATH,
    training_history_path,
]

missing = [str(path) for path in files_to_backup if not Path(path).is_file()]
if missing:
    raise FileNotFoundError(
        "Cannot create a complete backup; required files are missing:\n" + "\n".join(missing)
    )

with zipfile.ZipFile(BACKUP_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in files_to_backup:
        archive.write(path, arcname=Path(path).name)
        print("Added:", Path(path).name)

print("\nBackup created:", BACKUP_ZIP)
print("Size:", round(BACKUP_ZIP.stat().st_size / (1024**2), 2), "MB")

# FileLink must use a relative filename after changing to /kaggle/working.
os.chdir("/kaggle/working")
display(FileLink(
    "Experiment3_COMPLETE.zip",
    result_html_prefix="<b>Download the complete Experiment 3 backup: </b>"
))

## Notes for the project report

- Task: supervised image-to-image restoration, mapping a flare-corrupted RGB image to its clean RGB target.
- Data: Flickr24K clean backgrounds combined with Flare7K++ scattering and reflective flare patterns using the official on-the-fly loader; optional Flare-R patterns are loaded when present.
- Preprocessing and data split: same 256 × 256 transforms, 4,000 training backgrounds, 500 fixed validation backgrounds, and seeded split procedure as the earlier experiments.
- Architecture: compact Attention U-Net with four spatial attention gates on the encoder-to-decoder skip connections. It predicts a three-channel flare residual; clean estimate is input minus predicted residual.
- Initialisation: trained from scratch; no Baseline 1 or Experiment 2 weights are imported.
- Training objective: 0.7 ordinary L1 + 0.3 spatially weighted L1. The Experiment 2 low-flare preservation penalty is omitted because it explicitly suppresses corrections in low-residual pixels; this is a hypothesis-driven objective change, not proof that the term caused Experiment 2's weak-flare result.
- Checkpoint selection: ordinary L1 on the fixed validation set. Report PSNR, SSIM, full validation L1, weak/medium/strong residual-group L1, and visual inspection together.
- The weak/medium/strong groups are quantiles of mean absolute synthetic input-target residual on the fixed validation set. They are operational evaluation groups, not physical flare labels.
- Real captured images without clean ground truth should be assessed qualitatively; do not report reference-based PSNR/SSIM without a valid paired target.
- This experiment changes both architecture and loss objective relative to Experiment 2. Any gain cannot be attributed to attention gates alone.
- Recovery: full local checkpoint saved every 100 batches under `/kaggle/working/flare_unet_experiment_3`; a portable resume ZIP is refreshed every 500 batches and at each epoch boundary. The ZIP can be uploaded to Google Drive and restored in a new Kaggle session using `RESUME_BUNDLE_DRIVE_LINK`. Resume restores model, optimizer, scheduler, AMP scaler, RNG state, batch position, epoch permutation, and history.
- Backup: `Experiment3_COMPLETE.zip` contains the best weights, best checkpoint, latest resumable checkpoint, `results.json`, and `training_history.json`. Download it using the relative `FileLink` and copy it to Google Drive.
- Training budget for this run: up to 20 epochs, early-stopping patience 6; the actual number of epochs is recorded in `results.json`. The 4,000/500 split sizes are retained to avoid changing data volume at the same time as the architecture and loss objective.
